# RNN family from scratch -- vanilla RNN, LSTM, GRU

Three recurrent architectures, written in the order they were introduced
in the literature:

1. **Vanilla RNN** (Elman, 1990): the simplest possible recurrence.
   `h_t = tanh(W_hh h_{t-1} + W_hx x_t)`. Suffers from vanishing/exploding
   gradients on long sequences -- the motivation for the next two.
2. **LSTM** (Hochreiter & Schmidhuber, 1997): adds a separate cell state
   `c_t` and four gates (input, forget, output, candidate) that let
   gradient flow mostly unchanged through time when the forget gate is open.
3. **GRU** (Cho et al., 2014): a lighter alternative to LSTM -- merges the
   cell state into `h_t` and uses only two gates (reset, update).

All three are implemented with **manual loops over the time axis** so the
recurrence is explicit. PyTorch ships fused CUDA kernels (`nn.LSTM`,
`nn.GRU`) that are much faster -- I'm trading speed for transparency here.


## 0. Imports


In [46]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
import random
from tqdm import tqdm

import os
from torch.utils.tensorboard.writer import SummaryWriter
import numpy as np
import time
import math

## 1. Vanilla RNN

Multi-layer recurrence with per-layer weight matrices `W_hh` and `W_hx`. At
each time step `t`:

1. For each layer `l`, update `h^l_t = activation(W_hh^l h^l_{t-1} + W_hx^l x_t)`
   (using `x_t` directly for layer 0 and the previous layer's hidden state
   afterwards -- though in this implementation I keep the embedded input
   for every layer; that's a simplification, not strictly the stacked-RNN
   formulation).
2. Project the concatenated hidden states across layers to `output_size`
   via `W_yh`.

The `dropout` is applied to the hidden state between time steps.


In [49]:

class RNN(nn.Module) : # output_size should be vocab_size?
    def __init__(self, input_size, hidden_size, output_size, num_layers=1, activation = nn.Tanh, dropout=0.0):
        super(RNN, self).__init__()
        self.embedding = nn.Embedding(output_size , input_size)
        self.Wyh = nn.Linear(num_layers*hidden_size ,output_size ) 
        self.Whh = nn.ModuleList([nn.Linear(hidden_size, hidden_size) for _ in range(num_layers)])
        self.Whx  = nn.ModuleList([nn.Linear(input_size, hidden_size) for _ in range(num_layers)]) #nn.Linear(input_size, hidden_size*num_layers, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.activation = activation()
        # self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.num_layers = num_layers


    def forward(self , input , target = None)  :# (batch_size, seq_len (long))
        batch_size, seq_len = input.shape

        input = self.embedding(input)  # (batch_size, seq_len , input_size)

        h = torch.zeros((self.num_layers , batch_size,self.hidden_size) , device=input.device)  # (num_layers ,batch_size, hidden_size )

        output = torch.zeros((batch_size, seq_len , self.output_size), device = input.device) # (batch_size, seq_len , output_dim)
        for id in range(seq_len) :
            x = input[:,id] # (batch_size, input_size)
            
            for layer_id  in range(self.num_layers) :

                h[layer_id] = self.Whh[layer_id](h[layer_id]) + self.Whx[layer_id](x)   # (num_layers ,batch_size,hidden_size)

            h = self.activation(h) # (num_layers ,batch_size, hidden_dim)
            h = self.dropout(h)  # (num_layers , batch_size, hidden_dim)
            aggregation = h.transpose(0,1).contiguous().view(batch_size,-1) # (batch_size, hidden_dim*num_layers) 
            y = self.Wyh(aggregation)    # (batch_size, output_dim)
            output[:,id, :] = y # (batch_size, output_dim)

        if target is None :
            y = F.softmax(y , dim = -1) # (batch_size, seq_len , output_dim)
            loss = None

        else :
            output = output.view(batch_size*seq_len,-1)
            target = target.view(-1)
            loss = F.cross_entropy(output, target)

        return output, loss

### 1.1 Sanity check


In [50]:

model = RNN(64,16,31 , num_layers=5)
tensor1 = torch.randint(0,31,(8,5))
tensor2 = torch.randint(0,31,(8,5))

output,loss= model(tensor1,tensor2)
output.shape
print(loss.item())

3.4314284324645996


## 2. LSTM

Implementing the standard four-gate cell. I pack all four gate projections
into a **single** `nn.Linear(_, 4 * hidden_size)` for both the input-to-hidden
and hidden-to-hidden directions, then `chunk(4, dim=-1)` to split them. This
is the same trick `nn.LSTM` uses and is much faster than four separate
Linears.


In [ ]:
class LSTM(nn.Module):
    """Multi-layer LSTM written from scratch (Hochreiter & Schmidhuber, 1997).

    Each cell maintains a hidden state h and a *cell state* c. The four gates
    (input, forget, output, candidate) are computed from the concatenation of
    the previous hidden state and the current input:

        i_t = sigmoid(W_ii x_t + W_hi h_{t-1} + b_i)        # input gate
        f_t = sigmoid(W_if x_t + W_hf h_{t-1} + b_f)        # forget gate
        o_t = sigmoid(W_io x_t + W_ho h_{t-1} + b_o)        # output gate
        g_t = tanh(   W_ig x_t + W_hg h_{t-1} + b_g)        # candidate cell state
        c_t = f_t * c_{t-1} + i_t * g_t                     # cell state update
        h_t = o_t * tanh(c_t)                               # hidden state

    The gating allows information to flow through `c_t` mostly unchanged when
    `f_t` is near 1 and `i_t` is near 0, which is what mitigates the vanishing
    gradient problem of a vanilla RNN.
    """
    def __init__(self, input_size, hidden_size, output_size,
                 num_layers=1, dropout=0.0):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(output_size, input_size)
        # One Linear per layer producing all four gates concatenated (4 * hidden_size).
        self.W_x = nn.ModuleList([
            nn.Linear(input_size if l == 0 else hidden_size, 4 * hidden_size)
            for l in range(num_layers)
        ])
        self.W_h = nn.ModuleList([
            nn.Linear(hidden_size, 4 * hidden_size) for _ in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.W_out = nn.Linear(hidden_size, output_size)

    def forward(self, input, target=None):  # (B, T) long
        B, T = input.shape
        x_emb = self.embedding(input)  # (B, T, input_size)

        h = [torch.zeros(B, self.hidden_size, device=input.device) for _ in range(self.num_layers)]
        c = [torch.zeros(B, self.hidden_size, device=input.device) for _ in range(self.num_layers)]
        outputs = []

        for t in range(T):
            x = x_emb[:, t]  # (B, input_size)
            for l in range(self.num_layers):
                gates = self.W_x[l](x) + self.W_h[l](h[l])      # (B, 4*hidden)
                i, f, o, g = gates.chunk(4, dim=-1)
                i = torch.sigmoid(i)
                f = torch.sigmoid(f)
                o = torch.sigmoid(o)
                g = torch.tanh(g)
                c[l] = f * c[l] + i * g
                h[l] = o * torch.tanh(c[l])
                x = self.dropout(h[l])  # input to the next layer is this layer's hidden state
            outputs.append(self.W_out(h[-1]))  # (B, output_size)

        output = torch.stack(outputs, dim=1)  # (B, T, output_size)
        if target is None:
            return output, None
        loss = F.cross_entropy(output.reshape(B * T, -1), target.reshape(-1))
        return output, loss


# Smoke test
lstm = LSTM(input_size=64, hidden_size=16, output_size=31, num_layers=2)
inp = torch.randint(0, 31, (8, 5))
tgt = torch.randint(0, 31, (8, 5))
out, loss = lstm(inp, tgt)
print(f"LSTM output {out.shape}, loss {loss.item():.4f}")


## 3. GRU

Same packing trick as the LSTM, but with three gates instead of four. Note
that the **reset gate** is applied only to the `h -> n` projection, not to
the `x -> n` projection (paper Eq. 4) -- that's why I project `x` and `h`
separately rather than concatenating them first.


In [ ]:
class GRU(nn.Module):
    """Multi-layer GRU (Cho et al., 2014).

    Compared to LSTM, GRU merges the cell and hidden states into a single h_t
    and uses only **two** gates instead of four:

        r_t = sigmoid(W_ir x_t + W_hr h_{t-1} + b_r)            # reset gate
        z_t = sigmoid(W_iz x_t + W_hz h_{t-1} + b_z)            # update gate
        n_t = tanh(W_in x_t + r_t * (W_hn h_{t-1} + b_hn))      # candidate
        h_t = (1 - z_t) * n_t + z_t * h_{t-1}                   # blend

    The update gate z_t plays the role of the LSTM forget gate (controlling how
    much of the previous state to keep), while the reset gate r_t controls how
    much of the previous state contributes to the new candidate. Fewer
    parameters than LSTM (3 gates' worth instead of 4) and often comparable in
    practice on small/medium sequence tasks.
    """
    def __init__(self, input_size, hidden_size, output_size,
                 num_layers=1, dropout=0.0):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(output_size, input_size)
        # Per layer: project x to (r, z, n) and h to (r, z, n) separately so we
        # can apply the reset gate r to only the h->n projection (paper Eq. 4).
        self.W_x = nn.ModuleList([
            nn.Linear(input_size if l == 0 else hidden_size, 3 * hidden_size)
            for l in range(num_layers)
        ])
        self.W_h = nn.ModuleList([
            nn.Linear(hidden_size, 3 * hidden_size) for _ in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.W_out = nn.Linear(hidden_size, output_size)

    def forward(self, input, target=None):  # (B, T) long
        B, T = input.shape
        x_emb = self.embedding(input)
        h = [torch.zeros(B, self.hidden_size, device=input.device) for _ in range(self.num_layers)]
        outputs = []

        for t in range(T):
            x = x_emb[:, t]
            for l in range(self.num_layers):
                gx = self.W_x[l](x)          # (B, 3*hidden)
                gh = self.W_h[l](h[l])       # (B, 3*hidden)
                xr, xz, xn = gx.chunk(3, dim=-1)
                hr, hz, hn = gh.chunk(3, dim=-1)
                r = torch.sigmoid(xr + hr)
                z = torch.sigmoid(xz + hz)
                n = torch.tanh(xn + r * hn)
                h[l] = (1 - z) * n + z * h[l]
                x = self.dropout(h[l])
            outputs.append(self.W_out(h[-1]))

        output = torch.stack(outputs, dim=1)
        if target is None:
            return output, None
        loss = F.cross_entropy(output.reshape(B * T, -1), target.reshape(-1))
        return output, loss


# Smoke test
gru = GRU(input_size=64, hidden_size=16, output_size=31, num_layers=2)
out, loss = gru(inp, tgt)
print(f"GRU output {out.shape}, loss {loss.item():.4f}")
